In [1]:
from pyspark.sql import SparkSession

# Inisialisasi ulang session Spark
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession BERHASIL dinyalakan kembali!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 18:44:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession BERHASIL dinyalakan kembali!


In [3]:
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [4]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan4
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/mahasiswa/pertemuan4/
!hdfs dfs -ls /user/mahasiswa/pertemuan4/

put: `data_transaksi_ecommerce.csv': No such file or directory
Found 1 items
-rw-r--r--   1 azka supergroup      46827 2026-09-13 18:37 /user/mahasiswa/pertemuan4/data_transaksi_ecommerce.csv


In [2]:
df_dari_hdfs = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan4/data_transaksi_ecommerce.csv",
    header=True, inferSchema=True
)

print("Jumlah baris dari HDFS:", df_dari_hdfs.count())
df_dari_hdfs.show(5)

Jumlah baris dari HDFS: 600
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-05 00:00:00|          Elektronik|  Magelang|           8|      250000|     Kartu Kredit|
|ORD-1001|2026-07-08 00:00:00|        Rumah Tangga|      Solo|           5|      250000|         E-Wallet|
|ORD-1002|2026-07-27 00:00:00|Kesehatan & Kecan...|  Semarang|           9|      250000|     Kartu Kredit|
|ORD-1003|2026-07-31 00:00:00|        Rumah Tangga|Yogyakarta|           5|      500000|         E-Wallet|
|ORD-1004|2026-07-28 00:00:00|        Rumah Tangga|      Solo|           5|      250000|    Transfer Bank|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
only show

In [3]:
spark.stop()
print("SparkSession ditutup.")